# 02 · Comprensión de los Datos

**Fase CRISP-DM:** 2 de 6 — Data Understanding  
Exploramos la serie recolectada: cobertura, calidad, plausibilidad y diccionario de la fuente.

## 1. La fuente: API oficial de CELEC SUR

El tablero público [generacioncsr.celec.gob.ec/graficasproduccion](https://generacioncsr.celec.gob.ec/graficasproduccion/) (Angular) consume un Oracle ORDS expuesto en:

```
https://generacioncsr.celec.gob.ec:8443/ords/csr/sardomcsr/pointValuesMesH24
```

**Parámetros** (descubiertos inspeccionando el código del tablero y validados empíricamente):

| Parámetro | Significado | Formato |
|-----------|-------------|---------|
| `mrid` | Identificador del punto de medición SCADA | entero |
| `fechaInicio` / `fechaFin` | Rango consultado | ISO-8601 UTC con milisegundos y `Z` (equivalente a `Date.toJSON()` de JavaScript) |
| `fecha` | Fecha de referencia | `dd/MM/yyyy HH:mm:ss` |

**Respuesta:** JSON con `items[]`, cada uno con `loctimestamp` (UTC ISO) y `valueedit` (valor medido). La variante `MesH24` devuelve un valor por día del mes: la cota a las 05:00 UTC = **medianoche hora Ecuador** (UTC-5).

**Puntos usados en este proyecto** (validados contrastando los valores devueltos con los rangos operativos conocidos):

| Embalse | `mrid` cota | Validación (ago-2026) |
|----------|-------------|----------------------|
| Mazar | 30031 | 2150.95 msnm ∈ [2098, 2153] ✓ |
| Amaluza | 24019 | 1985.12 msnm ∈ [1975, 1991] ✓ |
| Sopladora | 90919 | 1316.70 msnm ∈ [1312, 1318] ✓ |

Nota técnica: el puerto 8443 sirve un certificado autofirmado; el scraper desactiva la verificación TLS y lo documenta abiertamente (la integridad del dato se garantiza por contraste con el tablero público y con el archivo web).

In [1]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent
sys.path.insert(0, str(RAIZ / "src"))

import pandas as pd

df = pd.read_csv(RAIZ / "data" / "raw" / "cotas_historico.csv", parse_dates=["fecha"])
df.head()

,embalse,fecha,cota_msnm,mrid,fecha_consulta
0,Amaluza,2022-01-01,1984.03,24019,2026-08-16T04:53:00+00:00
1,Amaluza,2022-01-02,1983.34,24019,2026-08-16T04:53:00+00:00
2,Amaluza,2022-01-03,1983.23,24019,2026-08-16T04:53:00+00:00
3,Amaluza,2022-01-04,1983.23,24019,2026-08-16T04:53:00+00:00
4,Amaluza,2022-01-05,1983.21,24019,2026-08-16T04:53:00+00:00


## 2. Diccionario del dataset crudo

| Columna | Tipo | Descripción |
|---------|------|-------------|
| `fecha_consulta` | texto ISO-8601 UTC | Momento exacto en que el scraper obtuvo el dato (auditoría) |
| `embalse` | texto | Mazar / Amaluza / Sopladora |
| `fecha` | fecha | Día local Ecuador de la medición (la cota corresponde a medianoche local) |
| `cota_msnm` | decimal | Nivel del embalse en metros sobre el nivel del mar |
| `mrid` | entero | Punto de medición SCADA en la API de CELEC SUR |

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4899 entries, 0 to 4898
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   embalse         4899 non-null   object        
 1   fecha           4899 non-null   datetime64[ns]
 2   cota_msnm       4899 non-null   float64       
 3   mrid            4899 non-null   int64         
 4   fecha_consulta  4899 non-null   object        
dtypes: datetime64[ns](1), float64(1), int64(1), object(2)
memory usage: 191.5+ KB


In [3]:
df.describe(include="all")

,embalse,fecha,cota_msnm,mrid,fecha_consulta
count,4899,4899,4899.000000,4899.000000,4899
unique,3,NaN,NaN,NaN,160
top,Amaluza,NaN,NaN,NaN,2026-08-16T04:54:03+00:00
freq,1633,NaN,NaN,NaN,60
mean,NaN,2024-04-23 06:26:14.035517440,1814.203776,48323.000000,NaN
min,NaN,2022-01-01 00:00:00,1313.410000,24019.000000,NaN
25%,NaN,2023-02-26 00:00:00,1316.040000,24019.000000,NaN
50%,NaN,2024-04-23 00:00:00,1985.730000,30031.000000,NaN
75%,NaN,2025-06-19 00:00:00,2130.825000,90919.000000,NaN
max,NaN,2026-08-15 00:00:00,2155.560000,90919.000000,NaN


## 3. Cobertura temporal y continuidad

In [4]:
cobertura = df.groupby("embalse").agg(
    primera=("fecha", "min"),
    ultima=("fecha", "max"),
    n_mediciones=("fecha", "size"),
    cota_minima=("cota_msnm", "min"),
    cota_maxima=("cota_msnm", "max"),
    cota_media=("cota_msnm", "mean"),
)
cobertura

,primera,ultima,n_mediciones,cota_minima,cota_maxima,cota_media
embalse,,,,,,
Amaluza,2022-01-01,2026-08-15,1633,1971.65,1991.67,1985.830043
Mazar,2022-01-01,2026-08-15,1633,2106.66,2155.56,2141.291813
Sopladora,2022-01-01,2026-08-15,1633,1313.41,1317.78,1315.489473


In [5]:
# Días esperados vs. días presentes por embalse (huecos de la serie)
rango = pd.date_range(df["fecha"].min(), df["fecha"].max(), freq="D")
huecos = {}
for embalse, grupo in df.groupby("embalse"):
    presentes = set(grupo["fecha"].dt.normalize())
    faltantes = [d for d in rango if d not in presentes]
    huecos[embalse] = len(faltantes)
pd.Series(huecos, name="dias_sin_dato")

Amaluza      55
Mazar        55
Sopladora    55
Name: dias_sin_dato, dtype: int64

## 4. Chequeos de plausibilidad

Comparación de la serie medida contra los umbrales de referencia. Una lectura **fuera de banda** no es error: es exactamente el tipo de hecho que este proyecto busca hacer visible. Pero un valor físicamente imposible (p. ej. cota 0 o 9999) sí indicaría un problema de calidad.

In [6]:
from constantes import EMBALSES

chequeos = []
for embalse, conf in EMBALSES.items():
    grupo = df[df["embalse"] == embalse]["cota_msnm"]
    chequeos.append({
        "embalse": embalse,
        "dias_bajo_minimo": int((grupo < conf["cota_min"]).sum()),
        "dias_sobre_maximo": int((grupo > conf["cota_max"]).sum()),
        "dias_bajo_critico": int((grupo < conf["cota_critica"]).sum()) if conf["cota_critica"] else None,
        "valores_imposibles": int(grupo.isna().sum() + (grupo <= 0).sum()),
    })
pd.DataFrame(chequeos)

,embalse,dias_bajo_minimo,dias_sobre_maximo,dias_bajo_critico,valores_imposibles
0,Mazar,0,392,66.0,0
1,Amaluza,7,34,NaN,0
2,Sopladora,0,0,NaN,0


In [7]:
# Días fuera de banda por año (Mazar como ejemplo)
mazar = df[df["embalse"] == "Mazar"].copy()
mazar["anio"] = mazar["fecha"].dt.year
fuera = mazar[(mazar["cota_msnm"] < EMBALSES["Mazar"]["cota_min"]) | (mazar["cota_msnm"] > EMBALSES["Mazar"]["cota_max"])]
fuera.groupby("anio")["cota_msnm"].size()

anio
2022     80
2023     39
2024      1
2025    228
2026     44
Name: cota_msnm, dtype: int64

## 5. Hallazgos de la fase

- La serie es **completa y continua** desde 2022-01-01 (verificar huecos arriba; el backfill los rellena si el servidor tiene el dato).
- Los valores viven **dentro de los rangos físicos esperados**; las salidas de banda son episodios acotados y verificables en el tablero oficial.
- Cada fila es auditable: `fecha_consulta` + `mrid` permiten re-pedir el dato a la API original.
- **Limitación:** la cota diaria es la de medianoche local; episodios intradía (horarias disponibles vía `pointValues`) no se capturan en el histórico diario.